*[Cell 1]*

# Chapter 8: Advanced Features

## Deletion Vectors Overview

### The Main Idea
Sometimes we can look at a problem and think of different ways to solve it. One of the ways we've done this in Delta Lake is to enable the feature called [deletion vectors](https://docs.delta.io/latest/delta-deletion-vectors.html). One of the things you will should consider in data design thinking about how you want to optimize the performance of a table. In essence, you can optimize for either the table readers or the table writers. It's also important to note that the term deletion vector defines more of the form and function of what the process does rather than how it helps you as a feature. One of the key benefits is that it gives you the ability to do a *merge-on-read* operation. It drastically reduces the performance implications of doing simple, delete operations and instead postpones the performance impact of those delete operations until a more convenient time in the future (but in a rather efficient manner).

### Merge-on-Read Explained
What does merge-on-read mean? Merge-on-read means that instead of going through the operation of rewriting a file at the time of deleting a record or set of records from a particular file we instead make some kind of a note that those records are deleted, and postpone the performance impact of actually performing the delete operation until a time where we can run and optimize command or more complicated update statement. Of course, if someone were to read the table after a merge-on-read operation has been initiated then it will merge during that read operation and that's kind of the point because it allows us to minimize the performance impacts of performing a simple delete operation, or even multiple delete operations, and just perform it at a time later where you are already filtering on the same set of files while reading them and otherwise avoid it doing so in situation's where you don't need it to happen straight away.
Deletion vectors are a way to get this kind of merge-on-read behavior. Put simply, deletion vectors are just a file (or multiple files) adjacent to a data file that allows you to know which records are to be deleted out of the larger data file it sits beside and push that delete (rewrite) operation off to a later point in time that is more efficient and convenient). Alongside in this case is relative, it is part of the Delta Lake file but you will notice in partitioned tables that the deletion vector files sit at the top level rather than within the partition. You can observe this in the coming example.

### Working Data
Our data originally hails from [this repo](https://github.com/nytimes/covid-19-data/issues) but is available in [the repo for this docker image](https://github.com/delta-io/delta-docker).

In [5]:
# [Cell 2]
# Databricks Free Edition: Spark runs on serverless via Databricks Connect (see dldg.py at the repo root)
from dldg import spark, DATASETS, WORK, set_conf, tree, rm, upload_dir
CH = f"{WORK}/ch08"  # replaces the Docker image's /opt/spark/work-dir

rm(f"{CH}/nyt_covid_19")

In [6]:
# [Cell 3]
# sc.setLogLevel isn't available over Databricks Connect

In [7]:
# [Cell 4]
spark.sql("drop table if exists workspace.dldg.nyt")

DataFrame[]

In [8]:
# [Cell 5]
from pyspark.sql.functions import col

(
spark
 .read
 .load(f"{DATASETS}/COVID-19_NYT")
 .filter(col("state")=="Florida")
 .filter(col("county").isin(['Hillsborough', 'Pasco', 'Pinellas', 'Sarasota']))
 .repartition("county")
 .write
 .format("delta")
 .partitionBy("county")
 .mode("overwrite")
 .save(f"{CH}/nyt_covid_19/")
 )

In [9]:
# [Cell 6]
(
spark
.read
.load(f"{CH}/nyt_covid_19/")
.write
.partitionBy("county")
.mode("overwrite")
.format("delta")
.saveAsTable("workspace.dldg.nyt")
)

In [10]:
# [Cell 7]
spark.sql("""
select
  date,
  county,
  state,
  count(1) as rec_count
from
  workspace.dldg.nyt
where
  county="Pinellas"
and
  date="2020-03-11"
group by
  date,
  county,
  state
order by
  date
""").show()

+----------+--------+-------+---------+
|      date|  county|  state|rec_count|
+----------+--------+-------+---------+
|2020-03-11|Pinellas|Florida|        1|
+----------+--------+-------+---------+



In [11]:
# [Cell 8]
spark.sql("ALTER TABLE workspace.dldg.nyt SET TBLPROPERTIES ('delta.enableDeletionVectors' = true);")

DataFrame[]

In [12]:
# [Cell 9]
# `workspace.dldg.nyt` is a Unity Catalog managed table, so its files can't be listed directly.
# DESCRIBE DETAIL shows the file count; `describe history` below shows deletion vectors added/removed.
spark.sql("describe detail workspace.dldg.nyt").select("numFiles", "sizeInBytes").show()

+--------+-----------+
|numFiles|sizeInBytes|
+--------+-----------+
|       4|      15743|
+--------+-----------+



In [13]:
# [Cell 10]
spark.sql("""
delete
from
  workspace.dldg.nyt
where
  county='Pinellas'
and
  date='2020-03-11'
""").show()

+-----------------+
|num_affected_rows|
+-----------------+
|                1|
+-----------------+



In [14]:
# [Cell 11]
# `workspace.dldg.nyt` is a Unity Catalog managed table, so its files can't be listed directly.
# DESCRIBE DETAIL shows the file count; `describe history` below shows deletion vectors added/removed.
spark.sql("describe detail workspace.dldg.nyt").select("numFiles", "sizeInBytes").show()

+--------+-----------+
|numFiles|sizeInBytes|
+--------+-----------+
|       4|      15743|
+--------+-----------+



In [15]:
# [Cell 12]
spark.sql("""
delete
from
  workspace.dldg.nyt
where
  county='Pasco'
""").show()

+-----------------+
|num_affected_rows|
+-----------------+
|              367|
+-----------------+



In [16]:
# [Cell 13]
spark.sql("""
delete
from
  workspace.dldg.nyt
where
  date='2020-03-13'
""").show()

+-----------------+
|num_affected_rows|
+-----------------+
|                3|
+-----------------+



In [17]:
# [Cell 14]
# `workspace.dldg.nyt` is a Unity Catalog managed table, so its files can't be listed directly.
# DESCRIBE DETAIL shows the file count; `describe history` below shows deletion vectors added/removed.
spark.sql("describe detail workspace.dldg.nyt").select("numFiles", "sizeInBytes").show()

+--------+-----------+
|numFiles|sizeInBytes|
+--------+-----------+
|       3|      12001|
+--------+-----------+



In [18]:
# [Cell 15]
spark.sql("describe history workspace.dldg.nyt").select("version", "operation", "operationMetrics").orderBy("version").show(truncate=False)

+-------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|operation                        |operationMetrics                                                                                                                                                                                                                                                                                                         |
+-------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

*[Cell 16]*


Notice that each one of our actions created a new table version, even when all we did was enable deletion vectors. If we look at the history results we can get a clearer picture of what happened at each step for the deletions. When we deleted just a single record from the table or multiple records from multiple files we can see in the operation metrics that we added a deletion vector to the table. When we deleted records aligned to entire partition the entire file was marked for removal and so we don't need a deletion vector to be added, it simply becomes stale and will be removed on the next clean up operation. When one of the deletion vectors was being applied to a table which already had a deletion vector being applied to it then it combines those results and marks the original for removal.

In [19]:
# [Cell 17]
# Each partition holds one small file, so OPTIMIZE has nothing to compact and leaves the deletion vectors alone
spark.sql("optimize workspace.dldg.nyt").select("metrics.numFilesAdded", "metrics.numFilesRemoved").show()

# REORG ... APPLY (PURGE) rewrites every file that has a deletion vector, physically dropping the deleted rows
spark.sql("reorg table workspace.dldg.nyt apply (purge)").select("metrics.numFilesAdded", "metrics.numFilesRemoved", "metrics.deletionVectorStats").show()
spark.sql("describe history workspace.dldg.nyt").select("version", "operation", "operationMetrics.numDeletionVectorsRemoved", "operationMetrics.numRemovedFiles", "operationMetrics.numAddedFiles").orderBy("version").show(truncate=False)


+-------------+---------------+
|numFilesAdded|numFilesRemoved|
+-------------+---------------+
|            0|              0|
+-------------+---------------+

+-------------+---------------+-------------------+
|numFilesAdded|numFilesRemoved|deletionVectorStats|
+-------------+---------------+-------------------+
|            3|              3|             {3, 4}|
+-------------+---------------+-------------------+

+-------+---------------------------------+-------------------------+---------------+-------------+
|version|operation                        |numDeletionVectorsRemoved|numRemovedFiles|numAddedFiles|
+-------+---------------------------------+-------------------------+---------------+-------------+
|0      |CREATE OR REPLACE TABLE AS SELECT|0                        |0              |NULL         |
|1      |SET TBLPROPERTIES                |NULL                     |NULL           |NULL         |
|2      |DELETE                           |0                        |0        

In [20]:
# [Cell 18]
set_conf("spark.databricks.delta.retentionDurationCheck.enabled", "false")

Skipped spark.databricks.delta.retentionDurationCheck.enabled=false: not settable on serverless compute


In [21]:
# [Cell 19]
# Serverless doesn't allow disabling spark.databricks.delta.retentionDurationCheck.enabled, so a
# 0-hour retention is rejected (DELTA_VACUUM_RETENTION_PERIOD_TOO_SHORT). Using the 7-day default instead.
spark.sql("vacuum workspace.dldg.nyt").show(truncate=False)

+----+
|path|
+----+
|    |
+----+



In [22]:
# [Cell 20]
# `workspace.dldg.nyt` is a Unity Catalog managed table, so its files can't be listed directly.
# DESCRIBE DETAIL shows the file count; `describe history` below shows deletion vectors added/removed.
spark.sql("describe detail workspace.dldg.nyt").select("numFiles", "sizeInBytes").show()

+--------+-----------+
|numFiles|sizeInBytes|
+--------+-----------+
|       3|      14572|
+--------+-----------+



*[Cell 21]*

## Constraints and Comments

In [25]:
# [Cell 22]
# Add transaction comments for successive operations
# (skipped on serverless; per write you can use .option("userMetadata", "...") instead)
set_conf("spark.databricks.delta.commitInfo.userMetadata", "setting-constraints-and-comments-example")

Skipped spark.databricks.delta.commitInfo.userMetadata=setting-constraints-and-comments-example: not settable on serverless compute


In [26]:
# [Cell 23]
spark.sql("drop table if exists workspace.dldg.example_table")

DataFrame[]

In [27]:
# [Cell 24]
spark.sql("""
create table workspace.dldg.example_table (
  id int comment 'uid column',
  content string not null comment 'payload column for text'
)
using delta
""")

DataFrame[]

In [28]:
# [Cell 25]
spark.sql("alter table workspace.dldg.example_table alter column id comment 'unique id column'")

DataFrame[]

In [29]:
# [Cell 26]
spark.sql("ALTER TABLE workspace.dldg.example_table ADD CONSTRAINT id CHECK (id > 0)")

DataFrame[]

In [30]:
# [Cell 27]
spark.sql("describe workspace.dldg.example_table").show( truncate=False)

+--------+---------+-----------------------+
|col_name|data_type|comment                |
+--------+---------+-----------------------+
|id      |int      |unique id column       |
|content |string   |payload column for text|
+--------+---------+-----------------------+



In [ ]:
# [Cell 28]
spark.sql("describe history workspace.dldg.example_table").select("version", "operation", "operationParameters", "userMetadata").show(truncate=False)

In [31]:
# [Cell 29]
spark.sql("show tblproperties workspace.dldg.example_table").show(truncate=False)

+--------------------------------------------------------+------------------------------------------------------------+
|key                                                     |value                                                       |
+--------------------------------------------------------+------------------------------------------------------------+
|delta.checkpointPolicy                                  |v2                                                          |
|delta.constraints.id                                    |id > 0                                                      |
|delta.enableDeletionVectors                             |true                                                        |
|delta.enableRowTracking                                 |true                                                        |
|delta.feature.appendOnly                                |supported                                                   |
|delta.feature.checkConstraints         

In [32]:
# [Cell 30]
# Remove annotations to log for successive operations
set_conf("spark.databricks.delta.commitInfo.userMetadata", "null")

Skipped spark.databricks.delta.commitInfo.userMetadata=null: not settable on serverless compute


In [33]:
# [Cell 31]
spark.sql("insert into workspace.dldg.example_table (id, content) values (1, 'thingmabob')")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [34]:
# [Cell 32]
# Uncomment the next line to test failing the constraint
spark.sql("insert into workspace.dldg.example_table (id, content) values (0, 'widget')")
# Fails with: [DELTA_VIOLATE_CONSTRAINT_WITH_VALUES] CHECK constraint id (id > 0) violated by row with values: - id : 0

UnknownException: (com.databricks.sql.transaction.tahoe.schema.DeltaInvariantViolationException) [DELTA_VIOLATE_CONSTRAINT_WITH_VALUES] CHECK constraint id (id > 0) violated by row with values:
 - id : 0.

JVM stacktrace:
com.databricks.sql.transaction.tahoe.schema.DeltaInvariantViolationException
	at com.databricks.sql.transaction.tahoe.schema.DeltaInvariantViolationException$.getConstraintViolationWithValuesException(InvariantViolationException.scala:82)
	at com.databricks.photon.PhotonException$.getSparkException(PhotonException.scala:552)
	at com.databricks.photon.PhotonWriteResultHandler.getResult(PhotonWriteStageExec.scala:153)
	at com.databricks.photon.PhotonBasicEvaluatorFactory$PhotonBasicEvaluator$$anon$1.open(PhotonBasicEvaluatorFactory.scala:299)
	at com.databricks.photon.PhotonBasicEvaluatorFactory$PhotonBasicEvaluator$$anon$1.hasNextImpl(PhotonBasicEvaluatorFactory.scala:304)
	at com.databricks.photon.PhotonBasicEvaluatorFactory$PhotonBasicEvaluator$$anon$1.$anonfun$hasNe

In [35]:
# [Cell 33]
# Uncomment the next line to test failing constraint
spark.sql("insert into workspace.dldg.example_table (id, content) values (2, null)")
# Fails with: [DELTA_NOT_NULL_CONSTRAINT_VIOLATED] NOT NULL constraint violated for column: content.

UnknownException: (com.databricks.sql.transaction.tahoe.schema.DeltaInvariantViolationException) [DELTA_NOT_NULL_CONSTRAINT_VIOLATED] NOT NULL constraint violated for column: content.


JVM stacktrace:
com.databricks.sql.transaction.tahoe.schema.DeltaInvariantViolationException
	at com.databricks.sql.transaction.tahoe.schema.DeltaInvariantViolationException$.getNotNullInvariantViolationException(InvariantViolationException.scala:57)
	at com.databricks.photon.PhotonException$.getSparkException(PhotonException.scala:541)
	at com.databricks.photon.PhotonWriteResultHandler.getResult(PhotonWriteStageExec.scala:153)
	at com.databricks.photon.PhotonBasicEvaluatorFactory$PhotonBasicEvaluator$$anon$1.open(PhotonBasicEvaluatorFactory.scala:299)
	at com.databricks.photon.PhotonBasicEvaluatorFactory$PhotonBasicEvaluator$$anon$1.hasNextImpl(PhotonBasicEvaluatorFactory.scala:304)
	at com.databricks.photon.PhotonBasicEvaluatorFactory$PhotonBasicEvaluator$$anon$1.$anonfun$hasNext$1(PhotonBasicEvaluato